# Day 09：Multi-Head Attention 实现与验证

本 notebook 把 `[B, S, D]` hidden states 经过 Q/K/V 投影、head 拆分、Scaled Dot-Product Attention、head 合并和输出投影的完整路径串起来。

学习目标：

1. 能写出 `[B,S,D] → [B,H,S,Dh] → [B,S,D]` 的 shape 变化；
2. 理解 `H=1` 为什么自然退化成单头 Attention；
3. 验证 `H>1` 与 PyTorch SDPA 数值对齐；
4. 用固定输入检查 head 合并顺序；
5. 验证 padding mask 与 `inference_mode`。

In [ ]:
import math

import torch
import torch.nn.functional as F

from mini_transformer import MultiHeadAttention
from mini_transformer.attention import build_padding_mask, naive_attention


torch.manual_seed(0)
print(f"PyTorch: {torch.__version__}")

## 1. 完整 shape 流程

设 `B=2`、`S=4`、`D=12`、`H=3`，则 `Dh=D/H=4`：

```text
hidden_states [2, 4, 12]
  → Wq/Wk/Wv
Q/K/V         [2, 4, 12]
  → view(B,S,H,Dh) + transpose(1,2)
Q/K/V         [2, 3, 4, 4]
  → Attention
每头输出       [2, 3, 4, 4]
  → transpose(1,2) + reshape(B,S,D)
合并输出       [2, 4, 12]
  → Wo
最终输出       [2, 4, 12]
```

拆头没有增加特征总量，因为 `H × Dh = D`。

In [ ]:
batch_size, sequence_length = 2, 4
hidden_size, num_heads = 12, 3

model = MultiHeadAttention(hidden_size=hidden_size, num_heads=num_heads)
hidden_states = torch.randn(batch_size, sequence_length, hidden_size)

projected_query = model.q_proj(hidden_states)
split_query = model._split_heads(projected_query)
output = model(hidden_states)

print("hidden_states: ", tuple(hidden_states.shape))
print("Wq output:     ", tuple(projected_query.shape))
print("split query:   ", tuple(split_query.shape))
print("MHA output:    ", tuple(output.shape))

assert split_query.shape == (2, 3, 4, 4)
assert output.shape == hidden_states.shape

## 2. split 与 merge 必须互为逆操作

拆分过程：

```text
[B,S,D] → view(B,S,H,Dh) → transpose(1,2) → [B,H,S,Dh]
```

合并过程必须先把 head 维换回去：

```text
[B,H,S,Dh] → transpose(1,2) → reshape(B,S,D) → [B,S,D]
```

`transpose` 后张量通常不连续，所以合并使用 `reshape`，而不是直接使用 `view`。只检查 shape 不够，必须检查所有元素是否回到原位置。

In [ ]:
ordered_states = torch.arange(2 * 4 * 12).reshape(2, 4, 12)
split_states = model._split_heads(ordered_states)
merged_states = model._merge_heads(split_states)

print("split shape: ", tuple(split_states.shape))
print("merged shape:", tuple(merged_states.shape))
print("逐元素一致:   ", torch.equal(merged_states, ordered_states))

assert torch.equal(merged_states, ordered_states)

## 3. `H=1` 自然退化为单头 Attention

`H=1` 时 `Dh=D`，仍然走相同的投影、拆头、Attention、合并和输出投影流程，不需要特殊分支。

把 `Wq/Wk/Wv/Wo` 都设为单位矩阵后，投影不改变输入，因此完整 MHA 应与直接调用 `naive_attention` 一致。

In [ ]:
single_head_model = MultiHeadAttention(hidden_size=4, num_heads=1)
identity = torch.eye(4)

with torch.no_grad():
    single_head_model.q_proj.weight.copy_(identity)
    single_head_model.k_proj.weight.copy_(identity)
    single_head_model.v_proj.weight.copy_(identity)
    single_head_model.out_proj.weight.copy_(identity)

single_head_input = torch.randn(2, 3, 4)
actual_single_head = single_head_model(single_head_input)

as_attention_input = single_head_input.unsqueeze(1)
expected_single_head = naive_attention(
    as_attention_input,
    as_attention_input,
    as_attention_input,
).squeeze(1)

single_head_error = (actual_single_head - expected_single_head).abs().max().item()
print(f"H=1 最大绝对误差: {single_head_error:.8f}")
torch.testing.assert_close(actual_single_head, expected_single_head)

## 4. `H>1` 与 PyTorch SDPA 对齐

这是 Day 9 的核心数值门禁。参考路径独立执行投影、reshape、transpose、SDPA、合并和 `Wo`，不调用 `_split_heads` / `_merge_heads`，避免实现和验证复制同一个 shape 错误。

这项实验能同时检查：

- Q/K/V 投影顺序；
- head 拆分与合并顺序；
- 缩放因子是否使用 `Dh`；
- softmax 是否沿 Key 维；
- 是否遗漏输出投影 `Wo`。

In [ ]:
torch.manual_seed(0)

batch_size, sequence_length = 2, 4
hidden_size, num_heads = 12, 3
head_dim = hidden_size // num_heads

multi_head_model = MultiHeadAttention(hidden_size=hidden_size, num_heads=num_heads)
multi_head_input = torch.randn(batch_size, sequence_length, hidden_size)
actual_multi_head = multi_head_model(multi_head_input)

query = multi_head_model.q_proj(multi_head_input).reshape(
    batch_size, sequence_length, num_heads, head_dim
).transpose(1, 2)
key = multi_head_model.k_proj(multi_head_input).reshape(
    batch_size, sequence_length, num_heads, head_dim
).transpose(1, 2)
value = multi_head_model.v_proj(multi_head_input).reshape(
    batch_size, sequence_length, num_heads, head_dim
).transpose(1, 2)

expected_heads = F.scaled_dot_product_attention(
    query,
    key,
    value,
    dropout_p=0.0,
    is_causal=False,
)
expected_merged = expected_heads.transpose(1, 2).reshape(
    batch_size, sequence_length, hidden_size
)
expected_multi_head = multi_head_model.out_proj(expected_merged)

multi_head_error = (actual_multi_head - expected_multi_head).abs().max().item()
print(f"H=3 与 SDPA 最大绝对误差: {multi_head_error:.8f}")
torch.testing.assert_close(actual_multi_head, expected_multi_head)

## 5. 固定输入手算：验证 head 合并顺序

使用 `D=4`、`H=2`、`Dh=2` 和单位投影。每个 head 中两个 token 的向量互相正交，因此缩放后的 scores 都是：

```text
[[1/√2, 0],
 [0, 1/√2]]
```

令 `a = softmax([1/√2, 0])[0]`、`b = softmax([1/√2, 0])[1]`，正确合并结果应为：

```text
token 0: [a, b, b, a]
token 1: [b, a, a, b]
```

如果跳过合并前的 `transpose(1, 2)`，shape 仍然正确，但元素顺序会错误。

In [ ]:
hand_model = MultiHeadAttention(hidden_size=4, num_heads=2)
identity = torch.eye(4)

with torch.no_grad():
    hand_model.q_proj.weight.copy_(identity)
    hand_model.k_proj.weight.copy_(identity)
    hand_model.v_proj.weight.copy_(identity)
    hand_model.out_proj.weight.copy_(identity)

hand_input = torch.tensor(
    [[[1.0, 0.0, 0.0, 1.0],
      [0.0, 1.0, 1.0, 0.0]]]
)

same_weight, other_weight = torch.softmax(
    torch.tensor([1 / math.sqrt(2), 0.0]),
    dim=0,
)
expected_hand = torch.tensor(
    [[[same_weight, other_weight, other_weight, same_weight],
      [other_weight, same_weight, same_weight, other_weight]]]
)
actual_hand = hand_model(hand_input)

print("a, b:", same_weight.item(), other_weight.item())
print("实际输出:\n", actual_hand)
print("期望输出:\n", expected_hand)
torch.testing.assert_close(actual_hand, expected_hand)

## 6. Padding mask 与 inference mode

验证 mask 最可靠的方法不是只观察概率，而是大幅修改被屏蔽位置的输入，再确认其他 Query 的输出不变。

注意 padding mask 屏蔽的是 PAD 位置作为 **Key/Value** 被其他 Query 读取；它不会删除 PAD 位置自己的 Query，所以只比较前三个有效 Query。

`torch.inference_mode()` 应放在模型外部的生成流程中，而不是写死在 `forward` 内，否则训练时无法构建计算图。

In [ ]:
torch.manual_seed(0)
mask_model = MultiHeadAttention(hidden_size=8, num_heads=2)
mask_input = torch.randn(1, 4, 8)
padding_mask = build_padding_mask(
    torch.tensor([[1, 1, 1, 0]]),
    dtype=mask_input.dtype,
)

modified_mask_input = mask_input.clone()
modified_mask_input[:, 3, :] += 1000

original_output = mask_model(mask_input, attn_mask=padding_mask)
modified_output = mask_model(modified_mask_input, attn_mask=padding_mask)
valid_query_error = (
    original_output[:, :3, :] - modified_output[:, :3, :]
).abs().max().item()

with torch.inference_mode():
    inference_output = mask_model(mask_input, attn_mask=padding_mask)

print(f"被屏蔽 token 改动后的有效 Query 最大误差: {valid_query_error:.8f}")
print("requires_grad:", inference_output.requires_grad)
print("is_inference: ", torch.is_inference(inference_output))

torch.testing.assert_close(original_output[:, :3, :], modified_output[:, :3, :])
assert inference_output.requires_grad is False
assert torch.is_inference(inference_output)

## 7. 五个常见错误与完成检查

1. **softmax 维度错误**：必须沿 `Skv`，即 `dim=-1`。
2. **拆头顺序错误**：必须先 `view(B,S,H,Dh)`，再 `transpose(1,2)`。
3. **mask 广播错误**：padding mask 必须是 `[B,1,1,Skv]`。
4. **缩放用了 `D`**：每个 head 应除以 `sqrt(Dh)`。
5. **合并顺序错误**：必须先 `transpose(1,2)`，再 `reshape(B,S,D)`。

完成标准：

- [x] `[B,S,D] → [B,H,S,Dh] → [B,S,D]` shape 正确；
- [x] split 与 merge 逐元素可逆；
- [x] `H=1` 与单头实现一致；
- [x] `H>1` 与 PyTorch SDPA 在 fp32 下对齐；
- [x] 固定输入的手算结果一致；
- [x] padding mask 生效；
- [x] inference mode 下不构建计算图。

对应实现：`src/mini_transformer/attention.py`  
对应测试：`tests/test_attention.py`